The goal for this file is twofold: 
1) To the connect the Immunization data from 2018 to 2025 as provided by the FSIS.
2) To tag each municipality as 1st to 5th income class as provided by the DOF

In [1]:
import pandas as pd
import numpy as np
from functools import reduce
import os 

pd.set_option('display.max_columns', None)


# 1. Connecting the Immunization Data from 2018 to 2025

From a first look through file system and the files within it, we know that:
* The granularity of the data vary based on the years. We have that:
    * For 2018 to 2024 data, it is annual, and we only have at most provincial- and city- level data.
    * For 2025 and 2026 data, the granularity is monthly and municipal-level. However, the 2026 data is only until February.
* The key performance indicators that the FHSIS reports are:
    * Fully Immunized Children (FIC) - The monthly target is 7.92%, which comes from the annual target of 95% divided by the 12 months.
        * According to the [DOH](https://doh.gov.ph/wp-content/uploads/2023/08/Booklet-7-Monitoring-Supportive-Supervision-and-Evaluation.pdf), for a child to be fully immunized, they need to have the following in the list. The FSIS also report the vaccination rates below:
            * One (1) dose of bCG at birth or anytime, 
            * Three (3) doses of OPV, 
            * Three (3) doses of Pentavalent vaccines; and 
            * One (1) dose of Measles-containing vaccine (MCV)
                * For this one, the FSIS records this as Two (2) doses of Measles Mumps Rubella (2 MMR) vaccine
    * Completely Immunized Children (CIC) - This one does not seem as important

Overall plan:
* Because of the differing granularities of the dataset, we will have two separate dataframes. They are:
    * Annual and provincial-level data from 2018 to 2025
    * Monthly and municipal-level data from 2025 to February 2026

For the naming schemes they are:

1. Annual Immunization Dataset – Column Definitions

| Column Name  | Description                                          |
| ------------ | ---------------------------------------------------- |
| Area         | Geographic area (e.g., region, province, or city)    |
| Eligible_Pop | Total number of individuals eligible for vaccination |

2. At Birth Indicators

| Column Name | Description                                                                                  |
| ----------- | -------------------------------------------------------------------------------------------- |
| CPAB        | Children Protected at Birth; infants protected against tetanus through maternal immunization |
| BCG         | Bacillus Calmette–Guérin vaccine; protects against tuberculosis                              |
| HEPA_B1     | Hepatitis B Birth Dose; first dose given at birth                                            |

3. Primary Series Vaccines

| Column Name | Description                                                             |
| ----------- | ----------------------------------------------------------------------- |
| DPT_1       | First dose of DPT-containing vaccine (Diphtheria, Pertussis, Tetanus)   |
| DPT_2       | Second dose of DPT-containing vaccine                                   |
| DPT_3       | Third dose of DPT-containing vaccine (completion of primary DPT series) |
| OPV_1       | First dose of Oral Polio Vaccine                                        |
| OPV_2       | Second dose of Oral Polio Vaccine                                       |
| OPV_3       | Third dose of Oral Polio Vaccine                                        |
| IPV         | Inactivated Polio Vaccine                                               |

4. Pneumococcal Vaccines

| Column Name | Description                                   |
| ----------- | --------------------------------------------- |
| PCV_1       | First dose of Pneumococcal Conjugate Vaccine  |
| PCV_2       | Second dose of Pneumococcal Conjugate Vaccine |
| PCV_3       | Third dose of Pneumococcal Conjugate Vaccine  |

5. Measles-Containing Vaccines

| Column Name | Description                               |
| ----------- | ----------------------------------------- |
| MCV_1       | First dose of Measles-Containing Vaccine  |
| MCV_2       | Second dose of Measles-Containing Vaccine |

6. Immunization Coverage Indicators

| Column Name | Description                                                                                                      |
| ----------- | ---------------------------------------------------------------------------------------------------------------- |
| FIC         | Fully Immunized Child; child who has received all recommended basic vaccines                                     |
| CIC         | Completely Immunized Child; broader definition depending on program (may include additional vaccines beyond FIC) |

---

Each vaccination indicator is further disaggregated and represented using the following suffix-based naming convention:

* `_M` for Male recipients
* `_F` for Female recipients
* `_Total` for Total recipients (Male + Female)
* `_Percent` for Proportion of vaccinated individuals relative to the eligible population

This means that for each base indicator (e.g., `DPT_1`, `BCG`, `FIC`), the dataset contains four corresponding columns:

* `{Indicator}_M`
* `{Indicator}_F`
* `{Indicator}_Total`
* `{Indicator}_Percent`

Changes of the FSIS in their measurements
* 2018 - did not measure IPV
* 2019 - 2021 - measured IPV only once
* 2022 - measured IPV 1 and IPV 2
* 2023 - started using somce the formal names as provided by the PSGC (so Baguio City instead of City of Baguio). However, other names like Manila City was still used
* 2024 - fully used the formal names provided by the PSGC (so City of Marikina) except for some cases like Pasay City (however most likely just a data encoding mistake, this will be fixed with regex)

* 2025 - municipality level data

## Cleaning the Data (80% of this notebook is just for cleaning.)

### Cleaning 2018 Data

Based on the table summary, we need the following tables:
* Table 2D.1	Proportion of FIC, CIC and Children Protected at Birth
* Table 2D.2	Proportion of Children given BCG and Hepatitis B1 Vaccines
* Table 2D.3	Proportion of Children given Pentavalent Vaccines
* Table 2D.4	Proportion of Children given Oral Polio Vaccines (OPV)
* Table 2D.5	Proportion of Children given Measles-containing Vaccines (MCV) and Rotavirus Vaccines


In [2]:
# Dealing with column names here 

TEMP_COLUMN_NAMES = [
 'Area',
 'CPAB',
 'BCG',
 'HEPA_B1',
 'DPT_1',
 'DPT_2',
 'DPT_3',
 'OPV_1',
 'OPV_2',
 'OPV_3',
 'IPV_1',
 'IPV_2',
 'PCV_1',
 'PCV_2',
 'PCV_3',
 'MCV_1',
 'MCV_2',
 'FIC',
 'CIC']

# Create column names
CODE_COLUMN_NAMES = TEMP_COLUMN_NAMES[:1]
for indicator in TEMP_COLUMN_NAMES[1:]:
    CODE_COLUMN_NAMES.append(indicator + '_M')
    CODE_COLUMN_NAMES.append(indicator + '_F')
    CODE_COLUMN_NAMES.append(indicator + '_Total')
    CODE_COLUMN_NAMES.append(indicator + '_Percent')

YEAR_CODE_COLUMN_NAMES = CODE_COLUMN_NAMES.copy()
YEAR_CODE_COLUMN_NAMES.insert(1, 'Year')

YEAR_PSGC_CODE_COLUMN_NAMES = (
    YEAR_CODE_COLUMN_NAMES[:1] + ['PSGC'] + YEAR_CODE_COLUMN_NAMES[1:]
)

def create_column_names(original_column_names, is_PSGC=False):
    """Add the _M, _F, _Total, _Percent to all indicators"""
    code_column_names = ['Area', 'Eligible_Pop']
    if is_PSGC:
        code_column_names = ['PSGC', 'Area', 'Eligible_Pop']

    for indicator in original_column_names:
        code_column_names.append(indicator + '_M')
        code_column_names.append(indicator + '_F')
        code_column_names.append(indicator + '_Total')
        code_column_names.append(indicator + '_Percent')

    return code_column_names

In [3]:
# 2018 data

sheet_filepath = 'data/medical/2018-2024/CC 2018.xlsx'
# TABLE 1
table_1_2018_df = pd.read_excel(sheet_filepath, 
                                   sheet_name='Table 2D.1',
                                   skiprows=7,
                                   skipfooter=3)

# get rid of the 10th column (number of live births) and the last column
table_1_2018_df = table_1_2018_df.drop(table_1_2018_df.columns[10], axis=1).copy()
table_1_2018_df = table_1_2018_df.drop(table_1_2018_df.columns[-1], axis=1).copy()

# rename columns
table_1_2018_df.columns = create_column_names(['FIC', 'CIC', 'CPAB'])
table_1_2018_df.dropna(inplace=True)

# TABLE 2
table_2_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.2',
                                skiprows=7, skipfooter=3)

# drop the last three columns
table_2_2018_df = table_2_2018_df.drop(table_2_2018_df.columns[-4:], axis=1).copy()
table_2_2018_df.columns = create_column_names(['BCG', 'HEPA_B1'])
table_2_2018_df.dropna(inplace=True)

# TABLE 3
table_3_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.3',
                                skiprows=7, skipfooter=3)

table_3_2018_df.columns = create_column_names(['DPT_1', 'DPT_2', 'DPT_3'])
table_3_2018_df.dropna(inplace=True)

# TABLE 4 
table_4_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.4',
                                skiprows=7, skipfooter=3)

table_4_2018_df.columns = create_column_names(['OPV_1', 'OPV_2', 'OPV_3'])
ipv_cols = create_column_names(['IPV_1', 'IPV_2'])[2:]
table_4_2018_df[ipv_cols] = 0
table_4_2018_df.dropna(inplace=True)

# TABLE 5
table_5_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.5',
                                skiprows=7, skipfooter=3)
table_5_2018_df = table_5_2018_df.drop(table_5_2018_df.columns[-8:], axis=1).copy()
table_5_2018_df.columns = create_column_names(['MCV_1', 'MCV_2'])
table_5_2018_df.dropna(inplace=True)

# TABLE 6
table_6_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.6',
                                skiprows=7,skipfooter=3)
table_6_2018_df.columns = create_column_names(['PCV_1', 'PCV_2', 'PCV_3'])
table_6_2018_df['Area'] = table_6_2018_df['Area'].replace('N C R 1', 'N C R')
table_6_2018_df.dropna(inplace=True)

table_dfs = [
    table_1_2018_df,
    table_2_2018_df,
    table_3_2018_df,
    table_4_2018_df,
    table_5_2018_df,
    table_6_2018_df
]

# remove Eligible_Pop column
for table_df in table_dfs:
    table_df.drop(columns='Eligible_Pop', inplace=True)

table_2018_df = reduce(
    lambda left, right: pd.merge(
        left, right,
        on=['Area'],
        how='inner'
    ),
    table_dfs
)

table_2018_df['Year'] = 2018

table_2018_df = table_2018_df[YEAR_CODE_COLUMN_NAMES].copy()

### Cleaning 2019 data

In [4]:
table_2019_df = pd.read_excel('data/medical/2018-2024/CC 2019.xlsx', 
                              sheet_name='Immunization',
                              skiprows=5, skipfooter=3).dropna()
table_2019_df.drop(columns=table_2019_df.columns[1], inplace=True)

# deal with the new IPV_2, so find where IPV_1 ends
insert_at = table_2019_df.columns.get_loc('%.9') + 1

# new IPV_2 columns
new_cols = [
    'IPV_2_M', 'IPV_2_F', 'IPV_2_Total', 'IPV_2_Percent'
]

# insert them in order
for i, col in enumerate(new_cols):
    table_2019_df.insert(insert_at + i, col, 0)   

table_2019_df.columns = CODE_COLUMN_NAMES
table_2019_df['Year'] = 2019
table_2019_df = table_2019_df[YEAR_CODE_COLUMN_NAMES].copy()

### Cleaning 2020 & 2021 data

The files for 2020 and 2021 have a semi-consistent format, so I will just use a function to extract the information. If there are any if statements, they are mainly for the weird files.

In [5]:
def create_table_df_version1(sheet_filepath, year):
    """Extract the data from 2020 to 2021"""
    # Table 1
    table1_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=0)
    table1_df = table1_df.iloc[:, :14].copy()
    table1_df.columns = create_column_names(['CPAB', 'BCG', 'HEPA_B1'])
    table1_df.dropna(inplace=True)

    # Table 2
    table2_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=1)
    table2_df.columns = create_column_names(['DPT_1', 'DPT_2', 'DPT_3'])
    table2_df.dropna(inplace=True)

    # Table 3
    table3_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=2)
    opv_ipv_columns = create_column_names(['OPV_1', 'OPV_2', 'OPV_3', 'IPV_1'])
    table3_df.columns = opv_ipv_columns
    ipv2_columns = create_column_names(['IPV_2'])[2:]
    table3_df[ipv2_columns] = 0
    table3_df.dropna(inplace=True)

    # Table 4
    table4_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=3)
    table4_df.columns = create_column_names(['PCV_1', 'PCV_2', 'PCV_3'])
    table4_df.dropna(inplace=True)

    # Table 5
    table5_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=4)
    mcv_only_cols = create_column_names(['MCV_1', 'MCV_2'])
    if len(table5_df.columns) != len(mcv_only_cols):
        table5_df.drop(columns=table5_df.columns[6], inplace=True)
    table5_df.columns = mcv_only_cols
    table5_df.dropna(inplace=True)

    # Table 6
    table6_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=4)
    fic_cic_only_cols = create_column_names(['FIC', 'CIC'])
    if len(table6_df.columns) != len(fic_cic_only_cols):
        table6_df.drop(columns=table6_df.columns[6], inplace=True)
    table6_df.columns = fic_cic_only_cols
    table6_df.dropna(inplace=True)

    table_dfs = [
        table1_df,
        table2_df,
        table3_df,
        table4_df,
        table5_df,
        table6_df
    ]

    for table_df in table_dfs:
        table_df.drop(columns='Eligible_Pop', inplace=True)

    table_df = reduce(
        lambda left, right: pd.merge(
            left, right,
            on=['Area'],
            how='inner'
        ),
        table_dfs
    )

    table_df['Year'] = year

    table_df = table_df[YEAR_CODE_COLUMN_NAMES]

    return table_df

In [6]:
table_2020_df = create_table_df_version1('data/medical/2018-2024/CC 2020.xlsx', 2020)

table_2021_df = create_table_df_version1('data/medical/2018-2024/CC 2021.xlsx', 2021)

### Cleaning 2022 to 2024 data

In [7]:
def create_table_df_version2(sheet_filepath, year, is_psgc):
    """Extract data from 2022 to 2024"""
    # Table 1
    table1_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=0)
    #table1_df = table1_df.iloc[:, :14].copy()
    if sheet_filepath.endswith("2024.xlsx"):
        table1_df.columns = create_column_names(['BCG', 'HEPA_B1', 'CPAB'], is_PSGC=is_psgc)
    else:
        table1_df.columns = create_column_names(['CPAB', 'BCG', 'HEPA_B1'], is_PSGC=is_psgc)
    table1_df.dropna(inplace=True)

    # Table 2
    table2_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=1)
    table2_df.columns = create_column_names(['DPT_1', 'DPT_2', 'DPT_3'], is_PSGC=is_psgc)
    table2_df.dropna(inplace=True)

    # # Table 3
    table3_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=2)
    opv_columns = create_column_names(['OPV_1', 'OPV_2', 'OPV_3'], is_PSGC=is_psgc)
    table3_df.columns = opv_columns
    table3_df.dropna(inplace=True)

    # Table 3.5 (IPV)
    table_ipv_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=3)
    if is_psgc:
        table_ipv_df = table_ipv_df.iloc[:, :11]
    else:
        table_ipv_df = table_ipv_df.iloc[:, :10]
    table_ipv_df.columns = create_column_names(['IPV_1', 'IPV_2'], is_PSGC=is_psgc)
    table_ipv_df.dropna(inplace=True)

    # Table 4
    table4_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=4)
    table4_df.columns = create_column_names(['PCV_1', 'PCV_2', 'PCV_3'], is_PSGC=is_psgc)
    table4_df.dropna(inplace=True)

    # Table 5
    table5_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=5)
    mcv_only_cols = create_column_names(['MCV_1', 'MCV_2'], is_PSGC=is_psgc)
    if len(table5_df.columns) != len(mcv_only_cols):
        table5_df.drop(columns=table5_df.columns[7], inplace=True)
    table5_df.columns = mcv_only_cols
    table5_df.dropna(inplace=True)

    # Table 6
    table6_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=6)
    fic_cic_only_cols = create_column_names(['FIC', 'CIC'], is_PSGC=is_psgc)
    if is_psgc:
        table6_df.drop(columns=table6_df.columns[7], inplace=True)
        
    if len(table6_df.columns) == len(fic_cic_only_cols) + 1:
        table6_df.drop(columns=table6_df.columns[6], inplace=True)
    
    table6_df.columns = fic_cic_only_cols
    table6_df.dropna(inplace=True)

    table_dfs = [
        table1_df,
        table2_df,
        table3_df,
        table_ipv_df,
        table4_df,
        table5_df,
        table6_df
    ]

    for table_df in table_dfs:
        table_df.drop(columns='Eligible_Pop', inplace=True)

    table_df['Year'] = year

    if is_psgc:
        table_df = reduce(
            lambda left, right: pd.merge(
                left, right,
                on=['Area', 'PSGC'],
                how='inner'
            ),
            table_dfs
        )

        PSGC_CODE_COLUMN_NAMES = YEAR_CODE_COLUMN_NAMES[:1] + ['PSGC'] + YEAR_CODE_COLUMN_NAMES[1:]
        return table_df[PSGC_CODE_COLUMN_NAMES].copy()

    else:
        table_df = reduce(
            lambda left, right: pd.merge(
                left, right,
                on=['Area'],
                how='inner'
            ),
            table_dfs
        )

        return table_df[YEAR_CODE_COLUMN_NAMES].copy()

In [8]:
table_2022_df = create_table_df_version2('data/medical/2018-2024/CC 2022.xlsx', 2022, False)
table_2023_df = create_table_df_version2('data/medical/2018-2024/CC 2023.xlsx', 2023, True)
table_2024_df = create_table_df_version2('data/medical/2018-2024/CC 2024.xlsx', 2024, False)

In [9]:
def combine_areas(
    df,
    areas_to_combine,
    new_area_name,
    new_psgc=None,   # optional
    year_col='Year',
    area_col='Area',
    psgc_col='PSGC'
):
    """
        Combine areas such as Maguindanao del Sur and 
        Maguindanao del Norte to one solidified province of Maguindanao.
        Makes sure that the percent is computed correctly and not just summed
    """
    df = df.copy()

    # Normalize Area names (prevents mismatch)
    df[area_col] = df[area_col]

    # Check if PSGC exists
    has_psgc = psgc_col in df.columns

    if has_psgc:
        df[psgc_col] = df[psgc_col].astype(str)

    # Filter rows to combine (AREA-BASED)
    subset_df = df[df[area_col].isin([a for a in areas_to_combine])]

    # Columns to sum
    exclude_cols = [area_col, year_col] + ([psgc_col] if has_psgc else [])
    value_cols = [col for col in df.columns if col not in exclude_cols]

    # Aggregate totals
    agg_df = subset_df.groupby(year_col)[value_cols].sum().reset_index()

    # Identify percent and total columns
    percent_cols = [col for col in value_cols if col.endswith('Percent')]
    tot_cols = [col for col in value_cols if col.endswith('Total')]

    # Recompute percent properly
    tot_df = subset_df[[year_col] + tot_cols].copy()

    percent_df_orig = subset_df[[year_col] + percent_cols].copy()
    percent_df_orig.columns = [year_col] + tot_cols  # align names

    eligible_df = (
        tot_df[tot_cols]
        .div(percent_df_orig[tot_cols].replace(0, np.nan))
        .mul(100)
    )

    tot_sum = tot_df.groupby(year_col)[tot_cols].sum()
    elig_sum = eligible_df.groupby(subset_df[year_col]).sum()

    percent_df = (
        tot_sum.div(elig_sum)
        .astype(float)
        .fillna(0)
    )

    percent_df.rename(
        columns=lambda col: col.replace('Total', 'Percent'),
        inplace=True
    )

    percent_df = percent_df.reset_index()

    # Replace percent columns
    agg_no_percent = agg_df.drop(columns=percent_df.columns[1:], errors='ignore')
    agg_df = agg_no_percent.merge(percent_df, on=year_col, how='left')

    # Add Area
    agg_df[area_col] = new_area_name

    # Add PSGC only if it exists
    if has_psgc:
        agg_df[psgc_col] = new_psgc

    # Reorder columns safely
    agg_df = agg_df[df.columns]

    # Remove original rows
    df = df[~df[area_col].isin([a for a in areas_to_combine])]

    # Append combined row
    df = pd.concat([df, agg_df], ignore_index=True)

    # Clean PSGC if present
    if has_psgc:
        df[psgc_col] = (
            df[psgc_col]
            .astype(str)
            .str.replace('.0', '', regex=False)
            .str.zfill(10)
        )
        df = df.sort_values(by=[psgc_col, year_col])
    else:
        df = df.sort_values(by=[area_col, year_col])

    return df

In [10]:
table_2024_df = combine_areas(
    table_2024_df,
    areas_to_combine=[
        'Maguindanao del Norte',
        'Maguindanao del Sur'
    ],
    new_area_name='Maguindanao'
)

### Cleaning 2025 Data

In [11]:
files = os.listdir('data/medical/2025')
excel_files = sorted([os.path.join('data/medical/2025', file) for file in files if file[0].isdigit()])

# TABLE 1
table1_df = pd.read_excel(
    excel_files[0], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)
table1_df.drop(columns=table1_df.columns[-5], inplace=True)

table1_df.columns = create_column_names(
    ['BCG_early', 'BCG_late', 'HEPA_B1_early', 'HEPA_B1_late', 'CPAB'],
    is_PSGC=True)

# BCG
table1_df['BCG_M'] = table1_df['BCG_early_M'] + table1_df['BCG_late_M']
table1_df['BCG_F'] = table1_df['BCG_early_F'] + table1_df['BCG_late_F']
table1_df['BCG_Total'] = table1_df['BCG_early_Total'] + table1_df['BCG_late_Total']
table1_df['BCG_Percent'] = table1_df['BCG_Total'] / table1_df['Eligible_Pop'] * 100

# HEPA_B1
table1_df['HEPA_B1_M'] = table1_df['HEPA_B1_early_M'] + table1_df['HEPA_B1_late_M']
table1_df['HEPA_B1_F'] = table1_df['HEPA_B1_early_F'] + table1_df['HEPA_B1_late_F']
table1_df['HEPA_B1_Total'] = table1_df['HEPA_B1_early_Total'] + table1_df['HEPA_B1_late_Total']
table1_df['HEPA_B1_Percent'] = table1_df['HEPA_B1_Total'] / table1_df['Eligible_Pop'] * 100
table1_df[['BCG_Percent', 'HEPA_B1_Percent']] = (
    table1_df[['BCG_Percent', 'HEPA_B1_Percent']].fillna(0)
)

relevant_cols = create_column_names(['CPAB', 'BCG', 'HEPA_B1'], is_PSGC=True)
table1_summarized_df = table1_df[relevant_cols].copy()

In [12]:
# TABLE 2
table2_df = pd.read_excel(
    excel_files[1], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table2_df = table2_df.iloc[:, :15]
table2_df.columns = create_column_names(['DPT_1', 'DPT_2', 'DPT_3'], is_PSGC=True)

# TABLE 3
table3_df = pd.read_excel(
    excel_files[2], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table3_df = table3_df.iloc[:, :15]
table3_df.columns = create_column_names(['OPV_1', 'OPV_2', 'OPV_3'], is_PSGC=True)

# TABLE 4
table4_df = pd.read_excel(
    excel_files[3], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table4_df = table4_df.iloc[:, :11]
table4_df.columns = create_column_names(['IPV_1', 'IPV_2'], is_PSGC=True)

# TABLE 5 
table5_df = pd.read_excel(
    excel_files[4], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table5_df = table5_df.iloc[:, :15]
table5_df.columns = create_column_names(['PCV_1', 'PCV_2', 'PCV_3'], is_PSGC=True)

# TABLE 6
table6_df = pd.read_excel(
    excel_files[5], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table6_df = table6_df.iloc[:, :11]
table6_df.columns = create_column_names(['MCV_1', 'MCV_2'], is_PSGC=True)

# TABLE 7 
table7_df = pd.read_excel(
    excel_files[6], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table7_df.drop(columns=table7_df.columns[7], inplace=True)
table7_df.columns = create_column_names(['FIC', 'CIC'], is_PSGC=True)

In [13]:
# merge all the 2025 data
table_dfs = [
    table1_df,
    table2_df,
    table3_df,
    table4_df,
    table5_df,
    table6_df,
    table7_df
]

table_dropped_dfs = [table_df.drop(columns=['Eligible_Pop']).copy() for table_df in table_dfs]

for table_dropped_df in table_dropped_dfs:
    table_dropped_df.Area = table_dropped_df.Area.replace(
        {"Maguindanao Sur" : "Maguindanao del Sur",
         "Maguindanao del Sur" : "Maguindanao del Sur"}
    )

# fix the fact that many municipalities can share the same name of Quezon
table_2025_df = reduce(
    lambda left, right: pd.merge(
        left, right.drop(columns=['Area']),
        on='PSGC',
        how='inner'
    ),
    table_dropped_dfs
)

# add Area back once
psgc_area_map = table_dropped_dfs[0][['PSGC', 'Area']].drop_duplicates()
table_2025_df = table_2025_df.drop(columns=['Area'], errors='ignore')
table_2025_df = table_2025_df.merge(psgc_area_map, on='PSGC', how='left')

table_2025_df['Year'] = 2025
table_2025_df = table_2025_df[YEAR_CODE_COLUMN_NAMES + ['PSGC']].copy()

# Deal with the Maguindanao problem
table_2025_df = combine_areas(
    table_2025_df,
    areas_to_combine=[
        'Maguindanao del Norte',
        'Maguindanao del Sur'
    ],
    new_area_name='Maguindanao',
    new_psgc='1908700000'
)

# fix two psgc
psgc_fixes = {
    "Zamboanga Sibugay": "0908300000",
    "Alburquerque": "0701201000",
}
for area, psgc in psgc_fixes.items():
    table_2025_df.loc[table_2025_df["Area"] == area, "PSGC"] = psgc

In [14]:
# Get the income
income_df = pd.read_excel('data/income/psgc-1q-2025-publication-datafile.xlsx', sheet_name='PSGC')
income_df = income_df[['10-digit PSGC', 'Income\nClassification']].copy()
income_df.columns = ['PSGC', 'Income']
income_df.PSGC = income_df.PSGC.astype(str).str.zfill(10)

# merge
table_2025_income_df = table_2025_df.merge(income_df, on='PSGC', how='left')

# I am adding this mainly for visualizations later on
table_2025_income_df.to_csv('outputs/annual2025_immunization.csv', index=False)


## Combining All of the Data from 2018 to 2025

1. Standardize Area names

   All `Area` values were first cleaned and made consistent across datasets. This included converting names to a uniform format (such as uppercase) and fixing small inconsistencies like extra spaces or different naming styles. This step ensures that areas match correctly when merging by name.

In [15]:
# standardized the alternative names
area_map = {
'A.R.M.M.': 'BARMM',
'Autonomous Region in Muslim Mindanao': 'BARMM',
'Compostela Valley': 'Davao de Oro',
'N C R' : 'NCR',
'C A R': 'CAR',
'Philippines':'PHILIPPINES',
'Northern Leyte':'Leyte',
'Mindoro Occidental':'Occidental Mindoro',
'Mindoro Oriental': 'Oriental Mindoro',
'Mt. Province':'Mountain Province',
'Province of Dinagat':'Dinagat Islands',
'Western Samar':'Samar',
'North Cotabato':'Cotabato',
'Maguindanao del Norte': 'Maguindanao',
'Maguindanao Del Norte': 'Maguindanao',
'Maguindanao Del Sur': 'Maguindanao',
'Maguindanao del Sur': 'Maguindanao',
'Gen. Santos City':'City of General Santos',
'CARAGA':'REGION 13'
}

updated_psgc_code = {
 604500000.0: 1804500000,
 630200000.0: 1830200000,
 704600000.0: 1804600000,
 706100000.0: 1806100000,
}


def standardize_area(area):
    """
        Standardize the area string by: 
        first, converting all the alternative names into the standard version using area_map;
        second, forcing old municipalities into cities;
        third, change the formatting of X City to City of X;
        finally, just make everything uppercase
    """
    area = area.strip()

    # standardize the names based on the mapping
    if area in area_map.keys():
        area = area_map[area]
        #print(repr(area))

    # force certain LGUs to be cities
    force_cities= ['Taguig', 'Malabon', 'Navotas', 'Olongapo', 'San Juan']
    if area in force_cities:
        return (f'City of {area}').upper()

    # convert "X City" -> "City of X"
    if area.endswith(' City'):
        name = area.replace(' City', '')
        return (f'City of {name}').upper()

    return area.upper()

2. Standardize PSGC codes

   The `PSGC` column was converted into string format to avoid issues caused by numeric types. Any decimal artifacts were removed, and values were formatted into 10-digit codes using zero-padding. This makes PSGC a reliable and consistent identifier across outside datasets.

In [16]:
# make sure that in the list, they are not just references
annual_tables_dfs = [
    table_2018_df,
    table_2019_df,
    table_2020_df,
    table_2021_df,
    table_2022_df,
    table_2023_df,
    table_2024_df,
    table_2025_df
]
annual_tables_dfs = [df.copy() for df in annual_tables_dfs]

# standardize all Area names first
for df in annual_tables_dfs:
    df['Area'] = df['Area'].apply(standardize_area)

# split datasets: up to 2024 vs 2025
tables_pre_2025 = [df.copy() for df in annual_tables_dfs[:-1]]
table_2025_df = annual_tables_dfs[-1].copy()

# make sure to update the PSGC
annual_tables_dfs[-3].PSGC = annual_tables_dfs[-3].PSGC.replace(updated_psgc_code)

# use 2023 as the reference for Area to PSGC
psgc_area_map = annual_tables_dfs[-3][['Area', 'PSGC']].copy()
psgc_area_map['PSGC'] = (
    psgc_area_map['PSGC']
    .astype(str)
    .str.replace('.0', '', regex=False)
    .str.zfill(10)  # ensures 10 digits with leading zeros
)

3. Use 2023 as the reference dataset

   The 2023 dataset was used as the base structure for the panel. It was chosen because it has the most stable and manageable number of entries (138 rows), compared to other years that are either more aggregated or more detailed. Using 2023 helps reduce mismatches when aligning data. For datasets from 2018 to 2024, merging was done using `Area`. During these years, the data is mostly at the provincial level, so duplicate names are not a problem. Each dataset was aligned to match the set of areas in 2023 before merging.

In [17]:
# combine 2018–2024 and align using 2023 Area names
annualized_pre_2025 = pd.concat(tables_pre_2025, ignore_index=True).drop(columns='PSGC')

annualized_pre_2025 = annualized_pre_2025.merge(
    psgc_area_map,
    on='Area',
    how='left' 
)

4. Merge 2025 using PSGC codes

   The 2025 dataset is more granular, with data at the municipal level. Because many municipalities share the same names, `Area` is no longer a reliable key. Instead, `PSGC` was used for merging. A right merge was performed using the PSGC codes from 2023 so that the final dataset still follows the same structure as the reference year.

In [18]:
# handle 2025 using PSGC as the key (more reliable for new splits)
table_2025_cleaned = table_2025_df.merge(
    psgc_area_map,
    on='PSGC',
    how='right'
)

# if Area columns duplicated, keep the 2023 version
if 'Area_y' in table_2025_cleaned.columns:
    table_2025_cleaned = (
        table_2025_cleaned
        .drop(columns=['Area_x'])
        .rename(columns={'Area_y': 'Area'})
    )


# combine everything into one panel dataset
annualized_completed_df = pd.concat(
    [annualized_pre_2025, table_2025_cleaned],
    ignore_index=True
)

annualized_completed_df = annualized_completed_df[YEAR_PSGC_CODE_COLUMN_NAMES].dropna().copy()

6. Handle administrative changes
   Some adjustments were needed to keep consistency across years:

* Maguindanao was split into Maguindanao del Norte and Maguindanao del Sur in 2025. These were combined back into a single “Maguindanao” entry by aggregating their values. When we tag the income classification for Maguindanao, we will use Maguindanao del Sur since the DOF identifies Maguindanao del Norte as not yet existing in 2008.
* For the Negros Island Region, the change mainly involved PSGC updates rather than structural differences, so codes were aligned and values combined where needed. Some areas (such as Negros Occidental, City of Bacolod, Negros Oriental, Siquijor, and Zamboanga Sibugay) had outdated PSGC codes in earlier datasets because of the NIR change. These were manually updated to their latest versions.

In [19]:
# add the nir region
# Ensure PSGC is string (important!)
annualized_completed_df['PSGC'] = annualized_completed_df['PSGC'].astype(str)

# Filter NIR (PSGC starting with '18')
nir_df = annualized_completed_df[
    annualized_completed_df['PSGC'].str.startswith('18')
]

# Columns to sum (exclude identifiers)
exclude_cols = ['Area', 'PSGC', 'Year']
value_cols = [col for col in annualized_completed_df.columns if col not in exclude_cols]

nir_agg = nir_df.groupby('Year')[value_cols].sum().reset_index()

# get the actual percent and tot cols
percent_cols = [
    value_col for value_col in value_cols 
    if value_col.endswith('Percent')
] + ['Year']

tot_cols = [
    value_col for value_col in value_cols 
    if value_col.endswith('Total')
] + ['Year']


# reconstruct the eligible population
tot_df = nir_df[tot_cols].copy()
percent_df = nir_df[percent_cols].copy()
percent_df.columns = tot_df.columns
eligible_df = (
    tot_df[tot_cols]
    .div(percent_df[tot_cols].replace(0, np.nan))
    .mul(100)
)

tot_sum = tot_df.groupby('Year')[tot_cols].sum()
elig_sum = eligible_df.groupby(nir_df['Year']).sum()

# get the actual ratio and rename
#percent_df = tot_sum.div(elig_sum).astype(float).fillna(0).mul(100).iloc[:, :-1]
percent_df = (
    tot_sum.div(elig_sum)
    .astype(float)
    .fillna(0)
    .mul(100)          # 👈 convert to percent
    .iloc[:, :-1]
    .clip(lower=0, upper=100)
)
renaming_map = {column : column.replace('Total', 'Percent') for column in percent_df.columns}
percent_df = percent_df.rename(columns=renaming_map).reset_index()

# combine the nir_agg with the updated percentages and add Area and PSGC info
nir_agg_no_percent = nir_agg.drop(columns=renaming_map.values())
nir_agg = nir_agg_no_percent.merge(percent_df, on="Year")
nir_agg['Area'] = 'NEGROS ISLAND REGION (NIR)'
nir_agg['PSGC'] = '1800000000'  # region-level PSGC (you can adjust if needed)
nir_agg = nir_agg[annualized_completed_df.columns]

# Append back
annualized_completed_df = pd.concat(
    [annualized_completed_df, nir_agg],
    ignore_index=True
)

# Now safe to sort
annualized_completed_df.sort_values(by=['PSGC', 'Year'], inplace=True)

# Tagging the Income Classification

With the Excel file provided about the PSGC codes, the tags on the income class are not actually provided. [DeepSeek](https://chat.deepseek.com/share/34sb7ym87whiiwjsj3) was used to convert [DOF PDF](https://blgf.gov.ph/wp-content/uploads/2025/01/DOF-DO-074.2024.pdf) into an Excel. Further cleaning will be done because it only has the names of the municipalities, and not the actual codes.

In [21]:
income_immunization_df = annualized_completed_df.merge(income_df, on=['PSGC'], how='left')

# SANITY CHECKS

We have three sanity checks to make sure that the merging of the two databases were correct and the values extracted are correct

1. First, check if the PSGC Codes agree that all the rows where they are supposed to be region are indeed regions. (Basically, the set of all rows such that the dataframe has the same element in the Area and Region column is equal to the set of all rows where its PSGC code has 8 trailing zeros.)

In [22]:
df = income_immunization_df.copy()
df['PSGC'] = df['PSGC'].astype(str).str.zfill(10)

# condition 1: Region
cond_name = df['Income'].isnull()

# condition 2: PSGC ends with 8 zeros
cond_psgc = df['PSGC'].str.endswith('00000000')

# check if they match for all rows
if (cond_name == cond_psgc).all():
    print("1st condition is passed! All the rows that are regions are indeed regions.")

1st condition is passed! All the rows that are regions are indeed regions.


2. Second, verify that each PSGC location appears eight times.

In [23]:
cond_consistent_appearance = (income_immunization_df.PSGC.value_counts() == 8).all()

if cond_consistent_appearance:
    print("2nd sanity check is passed! Each location appears 8 times.")

2nd sanity check is passed! Each location appears 8 times.


3. Third, we verified that all percentage-based columns exceeding 100% are consistent with the original data sources. A row-by-row validation against the source data confirms that these values are accurate. For example, the CIC Percentage for the City of Mandaluyong is indeed reported as 224% in the original dataset.

In [24]:
percent_cols = [col for col in income_immunization_df.columns if col.endswith('Percent')]

results = []

for col in percent_cols:
    idx = income_immunization_df[col].idxmax()
    
    results.append({
        'Metric': col,
        'Max Value': income_immunization_df.loc[idx, col],
        'Area': income_immunization_df.loc[idx, 'Area'],
        'Year': income_immunization_df.loc[idx, 'Year']
    })

max_summary_df = pd.DataFrame(results)
max_summary_df

,Metric,Max Value,Area,Year
0,CPAB_Percent,148.179881,BULACAN,2018
1,BCG_Percent,157.684864,CITY OF QUEZON,2019
2,HEPA_B1_Percent,191.557769,CITY OF ILOILO,2019
3,DPT_1_Percent,112.750271,CITY OF QUEZON,2023
4,DPT_2_Percent,110.168207,CAVITE,2020
5,DPT_3_Percent,150.377333,ZAMBOANGA SIBUGAY,2019
6,OPV_1_Percent,149.091414,CITY OF QUEZON,2019
7,OPV_2_Percent,143.979230,ILOCOS NORTE,2019
8,OPV_3_Percent,140.602151,CITY OF QUEZON,2019
9,IPV_1_Percent,106.993812,PATEROS,2019


# Exporting the Output File

In [25]:
income_immunization_df.Level = income_immunization_df.Level.fillna('Region')
income_immunization_df['Income'] = income_immunization_df['Income'].fillna('')

YEAR_PSGC_INCOME_CODE_COLUMN_NAMES = (
    YEAR_PSGC_CODE_COLUMN_NAMES[:2] + 
    ['Income'] + 
    YEAR_PSGC_CODE_COLUMN_NAMES[2:]
)

income_immunization_df = income_immunization_df[YEAR_PSGC_INCOME_CODE_COLUMN_NAMES].copy()

AttributeError: 'DataFrame' object has no attribute 'Level'

In [26]:
# Columns that should stay single (not grouped)
base_cols = ['Area', 'PSGC', 'Income', 'Year']


def write_with_grouped_headers(df, writer, sheet_name):
    df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1)
    
    workbook  = writer.book
    worksheet = writer.sheets[sheet_name]
    
    # Formats
    header_main = workbook.add_format({
        'bold': True,
        'align': 'center',
        'valign': 'middle',
        'border': 1,
        'bg_color': '#1D4E79',
        'font_color': '#FFFFFF'
    })

    header_sub = workbook.add_format({
        'bold': True,
        'align': 'center',
        'border': 1,
        'bg_color': '#1D4E79',
        'font_color': '#FFFFFF'
    })
        
    highlight_format = workbook.add_format({
        'bg_color': '#FFF2CC'  # light yellow
    })
    
    # ---- STEP 1: Handle base columns ----
    col_idx = 0
    for col in df.columns:
        if col in base_cols:
            worksheet.merge_range(0, col_idx, 1, col_idx, col, header_main)
            col_idx += 1
    
    # ---- STEP 2: Dynamically group remaining columns ----
    grouped_cols = [c for c in df.columns if c not in base_cols]
    
    # Extract prefix (before last "_")
    from collections import defaultdict
    groups = defaultdict(list)
    
    for col in grouped_cols:
        if '_' in col:
            prefix = '_'.join(col.split('_')[:-1])  # e.g. OPV_M → OPV
            groups[prefix].append(col)
        else:
            groups[col].append(col)
    
    # Write grouped headers
    for group, cols in groups.items():
        start_col = col_idx
        end_col = col_idx + len(cols) - 1
        
        # Top header
        worksheet.merge_range(0, start_col, 0, end_col, group, header_main)
        
        # Subheaders
        for i, col in enumerate(cols):
            sub = col.split('_')[-1] if '_' in col else col
            worksheet.write(1, start_col + i, sub, header_sub)
        
        col_idx += len(cols)
    
    # ---- STEP 3: Highlight rows where PSGC ends with 00000000 ----
    if 'PSGC' in df.columns:
        psgc_col_idx = df.columns.get_loc('PSGC')
        
        n_rows = len(df)
        n_cols = len(df.columns)
        
        # Helper to convert column index → Excel letter (handles AA, AB, etc.)
        def colnum_to_excel(n):
            string = ""
            while n >= 0:
                string = chr(n % 26 + 65) + string
                n = n // 26 - 1
            return string
        
        col_letter = colnum_to_excel(psgc_col_idx)
        
        highlight_format = workbook.add_format({
            'bg_color': '#A6A6A6',
            'bold': True
        })
    
    worksheet.conditional_format(
        2, 0,                     # start row, start col
        n_rows + 1, n_cols - 1,   # end row, end col
        {
            'type': 'formula',
            'criteria': f'=RIGHT(TEXT(${col_letter}3,"0"),8)="00000000"',
            'format': highlight_format
        }
    )
    
    # ---- STEP 4: Formatting ----
    worksheet.freeze_panes(2, 0)
    
    for i, col in enumerate(df.columns):
        worksheet.set_column(i, i, 18)


# ---- WRITE FILE ----
with pd.ExcelWriter('outputs/annual_immunization.xlsx', engine='xlsxwriter') as writer:
    
    # All data
    write_with_grouped_headers(income_immunization_df, writer, 'All_Data')
    
    # Per year
    years = sorted(income_immunization_df['Year'].unique())
    
    for year in years:
        yearly_df = income_immunization_df[
            income_immunization_df['Year'] == year
        ]
        
        write_with_grouped_headers(yearly_df, writer, str(year))

In [27]:
income_immunization_df.to_csv('outputs/annual_immunization.csv', index=False)